# Entrenamiento variacional

Todos los circuitos vistos hasta ahora fijan sus ángulos de antemano. Un circuito variacional deja algunos de esos ángulos como parámetros libres, y usa optimización clásica para ajustarlos según lo que se mide al ejecutarlo, hasta minimizar o maximizar una función de coste. Esa combinación de circuito cuántico y optimizador clásico es la base de QAOA y de otros algoritmos híbridos. Este notebook entrena un circuito QAOA pequeño para resolver MaxCut, el problema de encontrar el corte máximo de un grafo.

## Parámetros libres en un circuito

`polypus.Param(indice)` marca un ángulo como parámetro libre, en vez de un número fijo: es una referencia a una posición dentro del vector de parámetros que se bindea en el momento de ejecutar. `num_params` cuenta cuántos quedan pendientes:

In [ ]:
import polypus

qc_demo = polypus.Circuit(1)
qc_demo.rx(0, polypus.Param(0))
print(qc_demo.num_params)

Un circuito con parámetros pendientes no se puede ejecutar directamente:

In [ ]:
qc_demo.measure_all()
try:
    polypus.run_quantum_circuit(qc_demo, shots=100, infrastructure="local")
except ValueError as e:
    print(e)

El error mismo indica las dos formas de dejar de estar pendiente: pasar los valores a mano con `to_qasm2(params)`, o entrenarlos. El resto de este notebook hace lo segundo.

## El problema: MaxCut

MaxCut parte de un grafo, nodos unidos por aristas, y busca la forma de repartir los nodos en dos grupos que maximice el número de aristas con un extremo en cada grupo. Es un problema clásico de optimización combinatoria. Este notebook usa el caso más pequeño con más de una arista: un camino de tres nodos, con aristas entre el 0 y el 1, y entre el 1 y el 2.

In [ ]:
edges = [(0, 1), (1, 2)]


def cut_value(bitstring):
    bits = bitstring[::-1]  # bits[i] = valor del qubit i, como en 04a
    return sum(1 for i, j in edges if bits[i] != bits[j])


mejor_corte = 0
for a in (0, 1):
    for b in (0, 1):
        for c in (0, 1):
            mejor_corte = max(mejor_corte, cut_value(f"{a}{b}{c}"))

print(mejor_corte)

Con las tres aristas comprobadas a mano, el mejor corte posible es 2, por ejemplo separando el nodo 1 de los nodos 0 y 2.

## Construir el circuito: QAOA de una capa

QAOA parte de una superposición uniforme sobre las ocho particiones posibles, con una H en cada qubit. Después alterna dos capas: una capa de coste, que aplica una fase distinta según si los dos extremos de cada arista coinciden o difieren, con la puerta `rzz` vista en [`02a`](02a_puertas_y_construccion.ipynb), y una capa de mezcla, que redistribuye la amplitud entre particiones con una `rx` en cada qubit. Ambas capas comparten un único ángulo cada una, `gamma` para el coste y `beta` para la mezcla, los dos parámetros que se van a entrenar:

In [ ]:
Param = polypus.Param

qc = polypus.Circuit(3)
qc.h(0).h(1).h(2)  # superposicion uniforme sobre las 8 particiones

qc.rzz(0, 1, Param(0))  # capa de coste: una fase por cada arista
qc.rzz(1, 2, Param(0))

qc.rx(0, Param(1)).rx(1, Param(1)).rx(2, Param(1))  # capa de mezcla
qc.measure_all()

## Entrenar con `train()`

`polypus.train()` toma los mismos argumentos de infraestructura que `run_quantum_circuit`, más tres propios: `method`, el optimizador a usar, `dimensions`, el número de parámetros libres, y `expectation_function`, una función que recibe un único bitstring medido y devuelve su coste. Polypus se encarga de ejecutar el circuito, medir los shots, y promediar `expectation_function` sobre todos los bitstrings obtenidos, ponderado por cuántas veces salió cada uno: no hace falta promediar a mano. `cut_value`, ya definida, encaja tal cual.

El optimizador de esta sección es Evolución Diferencial, `DE`: mantiene una población de candidatos por generación, y los combina y muta buscando mejorar el mejor encontrado. `DE` maximiza la función de coste, así que un corte más grande es directamente un valor mejor, sin necesidad de invertir el signo.

In [ ]:
de = polypus.DE(generations=60, population_size=20, tolerance=0.001, seed=7)

result = polypus.train(
    qc,
    de,
    shots=1000,
    n_qpus=1,
    dimensions=qc.num_params,
    expectation_function=cut_value,
    infrastructure="local",
    nodes=1,
    cores_per_qpu=2,
    id="maxcut-de",
)
print(result.best_params)
print(result.best_fitness)

## Los campos de `TrainResult`

Además de `best_params` y `best_fitness`, el mejor punto encontrado y su coste, `result` trae otros campos útiles:

In [ ]:
print(result.iterations_run)
print(result.converged)
print(result.seed)
print(result.id)

`iterations_run` cuenta cuántas generaciones se ejecutaron de verdad: con 47 de las 60 posibles, `DE` paró antes de tiempo. `converged` lo confirma: `True` significa que la mejora acumulada de `best_fitness` bajó de `tolerance` antes de agotar el presupuesto, en vez de llegar al límite de generaciones sin estabilizarse. `seed` es la semilla efectiva usada, la pasada explícitamente en este caso, útil para reproducir exactamente esta ejecución. `id` es el identificador efectivo: Polypus añade un sufijo único al prefijo legible pasado en la llamada, para distinguir ejecuciones aunque compartan el mismo prefijo. El campo `id` del resultado, ya visto en [`03a_ejecucion_local.ipynb`](03a_ejecucion_local.ipynb) para `run_quantum_circuit`, sigue el mismo principio.

## Evaluar el resultado

`best_params` son solo números: para ejecutar el circuito con ellos hace falta bindearlos, igual que en el error de más arriba, con `to_qasm2(params)` seguido de `from_qasm2`:

In [ ]:
qc_final = polypus.Circuit.from_qasm2(qc.to_qasm2(result.best_params))
resultado_final = polypus.run_quantum_circuit(qc_final, shots=2000, infrastructure="local")
print(resultado_final.counts[0])

Con esos bitstrings se puede calcular el corte medio obtenido y compararlo con el mejor corte posible:

In [ ]:
counts = resultado_final.counts[0]
total = sum(counts.values())
corte_medio = sum(cut_value(bits) * n for bits, n in counts.items()) / total

print("corte medio medido:", corte_medio)
print("mejor corte posible:", mejor_corte)

El corte medio se queda por debajo del óptimo, alrededor del 82% de él: una sola capa de QAOA no garantiza encontrar el corte máximo, solo favorece las particiones con más aristas cruzadas frente a las demás. Más capas, con más parámetros que entrenar, acercarían el resultado al óptimo, a costa de un circuito más grande.

Con el mismo patrón visto en [`02b`](02b_interoperabilidad_qiskit_qasm.ipynb), se puede dibujar el circuito ya entrenado:

In [ ]:
from qiskit import qasm2

qc_dibujo = qasm2.loads(
    qc_final.to_qasm2(), custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS
)
qc_dibujo.draw("mpl")

## Resumen

Este notebook ha introducido los circuitos variacionales: ángulos marcados como parámetros libres con `Param`, entrenados con `train()` para maximizar una función de coste, usando Evolución Diferencial sobre un QAOA de una capa para MaxCut, y ha repasado los campos de `TrainResult`.

La siguiente sección repite este mismo problema cambiando de optimizador.

## Siguiente paso

Continúa con: [`06b_optimizadores_pso_qng.ipynb`](06b_optimizadores_pso_qng.ipynb).